# Model 1: Naive Baseline — Midfielders

**Can pre-transfer player qualities predict post-transfer player qualities?**

$$\text{to}_{Q_i} = \alpha + \beta \cdot \text{from}_{Q_i}$$

Each quality predicts **only itself** — 17 independent univariate regressions.

- **Input:** 1 feature per model (the same quality before transfer)
- **Output:** That quality after transfer
- **Position:** Midfielders only (17 qualities available)
- **Split:** Train 2018–2023, Test 2024

This is the simplest possible starting point: *"does a player stay roughly the same?"*

---

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import plotly.graph_objects as go
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

DATA = "../../../../thesis_data/processed_data/thesis_model_dataset/active/within_league_transfers_v5.parquet"
df = pd.read_parquet(DATA)
mf = df[df["from_position"] == "Midfielder"].copy()
print(f"Midfielders: {len(mf):,} rows")

Midfielders: 5,230 rows


In [2]:
# 17 qualities available for midfielders (3 excluded: Chance prevention, Territorial dominance, Poaching — >50% nulls)
QUALITIES = [
    "Active defence", "Aerial threat", "Box threat", "Composure",
    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",
    "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Winning duels",
]

from_cols = [f"from_{q}" for q in QUALITIES]
to_cols   = [f"to_{q}" for q in QUALITIES]

mf_clean = mf[from_cols + to_cols + ["from_season"]].dropna()
print(f"Complete rows: {len(mf_clean):,} / {len(mf):,} ({100*len(mf_clean)/len(mf):.1f}%)")

Complete rows: 5,159 / 5,230 (98.6%)


In [3]:
train = mf_clean[mf_clean["from_season"] <= 2023]
test  = mf_clean[mf_clean["from_season"] == 2024]
print(f"Train: {len(train):,} (2018–2023) | Test: {len(test):,} (2024)")

Train: 4,692 (2018–2023) | Test: 467 (2024)


## Example Input: Pre vs Post Transfer

Radar chart for one midfielder showing qualities before and after transfer.

In [4]:
ex = mf_clean.iloc[50]
from_vals = [ex[f"from_{q}"] for q in QUALITIES]
to_vals   = [ex[f"to_{q}"] for q in QUALITIES]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(r=from_vals + [from_vals[0]], theta=QUALITIES + [QUALITIES[0]],
    fill="toself", name="Pre-transfer", opacity=0.6, line=dict(color="#636EFA")))
fig.add_trace(go.Scatterpolar(r=to_vals + [to_vals[0]], theta=QUALITIES + [QUALITIES[0]],
    fill="toself", name="Post-transfer", opacity=0.6, line=dict(color="#EF553B")))
fig.update_layout(polar=dict(radialaxis=dict(visible=True)),
    title="Example Midfielder — Pre vs Post Transfer", height=550, template="plotly_white")
fig.show()

---
## Results

For each quality: `to_Qi = α + β · from_Qi` (simple linear regression).

In [5]:
results = []
models = {}

for q in QUALITIES:
    X_train = sm.add_constant(train[[f"from_{q}"]])
    X_test  = sm.add_constant(test[[f"from_{q}"]])
    y_train = train[f"to_{q}"]
    y_test  = test[f"to_{q}"]

    model = sm.OLS(y_train, X_train).fit()
    y_pred = model.predict(X_test)

    results.append({
        "Quality": q,
        "β": model.params.iloc[1],
        "β_pvalue": model.pvalues.iloc[1],
        "R²_train": model.rsquared,
        "R²_test": r2_score(y_test, y_pred),
        "MAE_test": mean_absolute_error(y_test, y_pred),
    })
    models[q] = model

res = pd.DataFrame(results)
print(res.to_string(index=False))
print(f"\nMean R²_train: {res['R²_train'].mean():.4f}")
print(f"Mean R²_test:  {res['R²_test'].mean():.4f}")
print(f"Mean MAE_test: {res['MAE_test'].mean():.4f}")

            Quality      β  β_pvalue  R²_train  R²_test  MAE_test
     Active defence 0.5716    0.0000    0.3299   0.2999    0.4463
      Aerial threat 0.5391    0.0000    0.2976   0.1867    0.4471
         Box threat 0.6631    0.0000    0.4101   0.3421    0.3685
          Composure 0.3127    0.0000    0.1056   0.0256    0.3937
  Defensive heading 0.6038    0.0000    0.3704   0.3576    0.4910
          Dribbling 0.5389    0.0000    0.3035   0.1895    0.3324
      Effectiveness 0.4609    0.0000    0.2006   0.1445    0.2564
          Finishing 0.1401    0.0000    0.0189   0.0135    0.5423
       Hold-up play 0.4459    0.0000    0.2161   0.1301    0.2665
Intelligent defence 0.5522    0.0000    0.3030   0.3037    0.4597
        Involvement 0.5612    0.0000    0.3088   0.3510    0.3702
    Passing quality 0.6810    0.0000    0.4679   0.4484    0.4086
           Pressing 0.5329    0.0000    0.2814   0.2883    0.4562
        Progression 0.6469    0.0000    0.4278   0.3815    0.4915
Providing 

### R²_test by Quality

How well does each quality predict itself after transfer?

In [6]:
fig = px.bar(res.sort_values("R²_test", ascending=True), x="R²_test", y="Quality",
    orientation="h", color="R²_test", color_continuous_scale="RdYlGn",
    range_color=[-0.05, 0.5])
fig.update_layout(title="R²_test — from_Qi predicts to_Qi (Midfielders)",
    height=500, template="plotly_white", showlegend=False)
fig.show()

### Full OLS Summary — Best and Worst

In [7]:
best_q = res.loc[res["R²_test"].idxmax(), "Quality"]
worst_q = res.loc[res["R²_test"].idxmin(), "Quality"]

print(f"BEST: {best_q} (R²_test = {res.loc[res['R²_test'].idxmax(), 'R²_test']:.4f})")
print(models[best_q].summary())
print(f"\n{'='*70}")
print(f"WORST: {worst_q} (R²_test = {res.loc[res['R²_test'].idxmin(), 'R²_test']:.4f})")
print(models[worst_q].summary())

BEST: Passing quality (R²_test = 0.4484)
                            OLS Regression Results                            
Dep. Variable:     to_Passing quality   R-squared:                       0.468
Model:                            OLS   Adj. R-squared:                  0.468
Method:                 Least Squares   F-statistic:                     4124.
Date:                Wed, 11 Mar 2026   Prob (F-statistic):               0.00
Time:                        16:50:50   Log-Likelihood:                -3588.1
No. Observations:                4692   AIC:                             7180.
Df Residuals:                    4690   BIC:                             7193.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------

### Scatter: Best vs Worst Quality

In [8]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=[f"BEST: {best_q}", f"WORST: {worst_q}"])

for i, q in enumerate([best_q, worst_q], 1):
    fig.add_trace(go.Scatter(
        x=test[f"from_{q}"], y=test[f"to_{q}"], mode="markers",
        marker=dict(size=5, opacity=0.5), name=q, showlegend=False
    ), row=1, col=i)
    # Add y=x line
    mn, mx = test[f"from_{q}"].min(), test[f"from_{q}"].max()
    fig.add_trace(go.Scatter(x=[mn, mx], y=[mn, mx], mode="lines",
        line=dict(dash="dash", color="red"), name="y=x", showlegend=(i==1)
    ), row=1, col=i)
    fig.update_xaxes(title_text=f"from_{q}", row=1, col=i)
    fig.update_yaxes(title_text=f"to_{q}", row=1, col=i)

fig.update_layout(height=400, template="plotly_white",
    title="Pre vs Post Transfer (test set) — red dashed = no change")
fig.show()

---
## Takeaway

**β < 1 siempre** → regression to the mean. Los jugadores extremos (muy buenos o muy malos) tienden a acercarse al promedio después de transferirse.

**R²_test promedio ~25%** → la quality antes del transfer explica ~25% de la variación de la quality después. Es un baseline decente pero deja mucho sin explicar.

**Mejores:** Passing quality, Progression, Run quality (~37-45% R²_test)
**Peores:** Finishing, Composure (~1-3% R²_test) — muy volátiles entre temporadas.

**Siguiente modelo:** Agregar las 17 qualities como features (cross-quality effects).